# ACTO SDK: Async Proof Operations

This notebook demonstrates async/await support for proof creation and verification.


In [ ]:
import asyncio
from datetime import datetime

from acto.crypto import KeyPair
from acto.proof import create_proof_async, verify_proof_async
from acto.telemetry.models import TelemetryBundle, TelemetryEvent


## Async Proof Creation


In [ ]:
async def create_proof_example():
    # Generate keypair
    keypair = KeyPair.generate()
    
    # Create telemetry bundle
    bundle = TelemetryBundle(
        task_id="async-task-001",
        robot_id="robot-001",
        events=[
            TelemetryEvent(
                ts=datetime.now().isoformat(),
                topic="imu",
                data={"accel": [1.0, 2.0, 3.0]}
            )
        ]
    )
    
    # Create proof asynchronously
    envelope = await create_proof_async(
        bundle,
        keypair.private_key_b64,
        keypair.public_key_b64
    )
    
    print(f"Proof created: {envelope.payload.payload_hash}")
    return envelope

# Run async function
envelope = await create_proof_example()


## Async Proof Verification


In [ ]:
async def verify_proof_example():
    # Verify proof asynchronously
    is_valid = await verify_proof_async(envelope)
    print(f"Proof is valid: {is_valid}")

await verify_proof_example()


## Batch Async Operations


In [ ]:
async def batch_create_proofs():
    keypair = KeyPair.generate()
    
    # Create multiple proofs concurrently
    tasks = []
    for i in range(5):
        bundle = TelemetryBundle(
            task_id=f"task-{i:03d}",
            robot_id="robot-001",
            events=[
                TelemetryEvent(
                    ts=datetime.now().isoformat(),
                    topic="sensor",
                    data={"data": i}
                )
            ]
        )
        tasks.append(
            create_proof_async(
                bundle,
                keypair.private_key_b64,
                keypair.public_key_b64
            )
        )
    
    # Wait for all proofs to be created
    envelopes = await asyncio.gather(*tasks)
    print(f"Created {len(envelopes)} proofs")
    
    # Verify all proofs
    verify_tasks = [verify_proof_async(env) for env in envelopes]
    results = await asyncio.gather(*verify_tasks)
    print(f"All proofs valid: {all(results)}")

await batch_create_proofs()
